# 02 — Baseline Classifier

**Goal:** establish baseline performance using *all* features, before any feature
selection is applied. Every metaheuristic notebook (GA/PSO/GWO/WOA) will be judged
against this baseline in `07_Comparison.ipynb` — if a feature-selection algorithm
can't beat (or at least match) this baseline while using fewer features, it isn't
adding value.

We evaluate two classifiers (SVM and Random Forest) using the exact same
`evaluate_subset()` function that GA/PSO/GWO/WOA will use later, so the comparison
is apples-to-apples.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "utils").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    ConfusionMatrixDisplay,
    roc_curve,
)
from sklearn.model_selection import StratifiedShuffleSplit

from utils.config import (
    DATASETS,
    CLASSIFIERS,
    CLASSIFIER_SEED,
    USE_XGB_GPU,
    RESULTS_DIR,
    PLOTS_DIR,
)
from utils.preprocessing import load_processed_data
from utils.metrics import (
    get_classifier,
    evaluate_subset,
    get_confusion_matrix,
)

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

fractions = [0.20, 0.40, 0.60, 0.80, 1.00]

for dataset_name in DATASETS:
    (
        X_train,
        X_val,
        X_test,
        y_train,
        y_val,
        y_test,
        feature_names,
    ) = load_processed_data(dataset_name)

    fig, axes = plt.subplots(
        1,
        len(CLASSIFIERS),
        figsize=(16, 4),
        sharey=True,
    )

    for ax, classifier_name in zip(
        axes, CLASSIFIERS
    ):
        train_scores = []
        validation_scores = []
        sample_counts = []

        for fraction in fractions:
            if fraction == 1.0:
                X_subset = X_train
                y_subset = y_train
            else:
                splitter = StratifiedShuffleSplit(
                    n_splits=1,
                    train_size=fraction,
                    random_state=CLASSIFIER_SEED,
                )

                subset_indices, _ = next(
                    splitter.split(X_train, y_train)
                )

                X_subset = X_train[subset_indices]
                y_subset = y_train[subset_indices]

            model = get_classifier(
                classifier_name,
                random_state=CLASSIFIER_SEED,
                use_gpu=(
                    USE_XGB_GPU
                    and classifier_name == "xgboost"
                ),
            )

            model.fit(X_subset, y_subset)

            train_scores.append(
                accuracy_score(
                    y_subset,
                    model.predict(X_subset),
                )
            )
            validation_scores.append(
                accuracy_score(
                    y_val,
                    model.predict(X_val),
                )
            )
            sample_counts.append(len(y_subset))

        ax.plot(
            sample_counts,
            train_scores,
            marker="o",
            label="Training",
        )
        ax.plot(
            sample_counts,
            validation_scores,
            marker="o",
            label="Validation",
        )
        ax.set_title(classifier_name)
        ax.set_xlabel("Training samples")
        ax.set_ylabel("Accuracy")
        ax.set_ylim(0.5, 1.03)
        ax.grid(alpha=0.3)
        ax.legend()

    plt.suptitle(
        f"{dataset_name}: training vs validation"
    )
    plt.tight_layout()
    plt.savefig(
        PLOTS_DIR
        / f"{dataset_name}_learning_curves.png",
        dpi=150,
    )
    plt.show()

In [ ]:
baseline_rows = []

for dataset_name in DATASETS:
    (
        X_train,
        X_val,
        X_test,
        y_train,
        y_val,
        y_test,
        feature_names,
    ) = load_processed_data(dataset_name)

    X_development = np.vstack([X_train, X_val])
    y_development = np.concatenate([y_train, y_val])

    full_mask = np.ones(
        X_development.shape[1],
        dtype=int,
    )

    evaluation_cache = {}

    for classifier_name in CLASSIFIERS:
        result = evaluate_subset(
            full_mask,
            X_development,
            X_test,
            y_development,
            y_test,
            classifier=classifier_name,
            random_state=CLASSIFIER_SEED,
            use_gpu=(
                USE_XGB_GPU
                and classifier_name == "xgboost"
            ),
        )

        evaluation_cache[classifier_name] = result

        baseline_rows.append({
            "Dataset": dataset_name,
            "Classifier": classifier_name,
            "Algorithm": "Baseline",
            "Seed": CLASSIFIER_SEED,
            "ValidationFitness": np.nan,
            "Accuracy": result["accuracy"],
            "Precision": result["precision"],
            "Recall": result["recall"],
            "F1": result["f1"],
            "ROC_AUC": result["roc_auc"],
            "Features": result["n_features"],
            "SelectionRuntime": 0.0,
            "TestRuntime": result["runtime"],
            "SelectedFeatureNames": json.dumps(
                feature_names
            ),
            "MaskFile": "",
            "ConvergenceFile": "",
        })

    # Test ROC curves
    plt.figure(figsize=(6, 5))

    for classifier_name, result in (
        evaluation_cache.items()
    ):
        fpr, tpr, _ = roc_curve(
            y_test,
            result["y_score"],
        )

        plt.plot(
            fpr,
            tpr,
            label=(
                f"{classifier_name} "
                f"(AUC={result['roc_auc']:.3f})"
            ),
        )

    plt.plot([0, 1], [0, 1], "--", color="gray")
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.title(f"{dataset_name}: baseline ROC")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(
        PLOTS_DIR / f"{dataset_name}_baseline_roc.png",
        dpi=150,
    )
    plt.show()

baseline_df = pd.DataFrame(baseline_rows)

baseline_df.to_csv(
    RESULTS_DIR / "baseline_results.csv",
    index=False,
)

baseline_df

In [ ]:
for dataset_name in DATASETS:
    (
        X_train,
        X_val,
        X_test,
        y_train,
        y_val,
        y_test,
        feature_names,
    ) = load_processed_data(dataset_name)

    model = get_classifier(
        "xgboost",
        random_state=CLASSIFIER_SEED,
        use_gpu=USE_XGB_GPU,
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[
            (X_train, y_train),
            (X_val, y_val),
        ],
        verbose=False,
    )

    history = model.evals_result()

    plt.figure(figsize=(7, 4))
    plt.plot(
        history["validation_0"]["logloss"],
        label="Training loss",
    )
    plt.plot(
        history["validation_1"]["logloss"],
        label="Validation loss",
    )
    plt.xlabel("Boosting round")
    plt.ylabel("Log loss")
    plt.title(
        f"{dataset_name}: XGBoost learning history"
    )
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(
        PLOTS_DIR
        / f"{dataset_name}_xgboost_loss.png",
        dpi=150,
    )
    plt.show()